In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os
import numpy as np
import pandas as pd
import scanpy as sc
#import STALocator
import scipy.sparse as sp
import torch
from datetime import datetime
from scad import *
experiment_path = "result/mouse"
model_path = os.path.join(experiment_path, "models")
data_path = os.path.join(experiment_path, "data")
result_path = os.path.join(experiment_path, "results")
os.makedirs(model_path, exist_ok=True)
os.makedirs(data_path, exist_ok=True)
os.makedirs(result_path, exist_ok=True)
adata_ST=sc.read_h5ad('results/enhanced_exp_MB.h5ad')
adata_sc=sc.read('data/adata_sc_MB.h5ad')

In [ ]:
adata_ST.obsm['spatial']=adata_ST.obs[['x','y']].to_numpy()
svg_list=pd.read_csv('results/svg-enhanced-MB.csv',header=0,sep=',',index_col=0)
svg_list=svg_list.index
adata_ST.var_names_make_unique()
adata_sc.var_names_make_unique()

## Simulation

In [ ]:
import numpy as np

def replace_far_samples(X, R1,R2, ratio=0.05, random_state=None):
    """
    Randomly select `ratio` of samples and move them to another sample's location
    that is at a distance greater than R from the original point.

    Parameters:
        X: numpy array, shape (N, 2)
        R: float, distance threshold
        ratio: float, proportion of samples to select (default 0.05)
        random_state: int or None, seed for reproducibility

    Returns:
        X: modified array
        selected_idx: indices of selected samples
        to_index: corresponding target indices for each selected sample
    """
    rng = np.random.default_rng(random_state)
    N = X.shape[0]

    # Step 1: randomly select `ratio` of samples
    num_samples = max(1, int(N * ratio))
    selected_idx = rng.choice(N, size=num_samples, replace=False)
    to_index = selected_idx.copy()

    # Step 2: for each selected sample, find another point farther than R
    j = -1
    for i in selected_idx:
        j += 1
        point = X[i]
        diffs = X - point
        dists = np.linalg.norm(diffs, axis=1)
        q10 = np.quantile(dists, R1)   
        q20 = np.quantile(dists, R2)  
        # exclude self, keep points with distance > R
        valid_idx = np.where((dists > R1) & (np.arange(N) != i)&(dists < R2))[0]

        if len(valid_idx) > 0:
            chosen = rng.choice(valid_idx)
            X[i] = X[chosen]
            to_index[j] = chosen

    return X, selected_idx, to_index

In [11]:
adata_ST.obsm['spatial'] = (adata_ST.obsm['spatial']-adata_ST.obsm['spatial'].min(axis=0))/(adata_ST.obsm['spatial'].max(axis=0)-adata_ST.obsm['spatial'].min(axis=0))
adata_ST.obsm['spatial'] ,selected_idx,to_index= replace_far_samples(adata_ST.obsm['spatial'],0.1,0.2, ratio=0.05,random_state=12306)
N_st =adata_ST.shape[0]
splits = split_dataset_cv3(N_st, seed=42)
Y=adata_ST.obs[['x','y']].values
Y = Y.astype(np.float32)
N_rna=adata_sc.X.shape[0]
training_idx_rna=np.array(range(N_rna))

In [12]:
    model4 = Model3(
    resolution="low",
    batch_size=200,
    train_epoch=3000,
    cut_steps=0.5,
    sf_coord=50,
    rad_cutoff=1.2,
    seed=1234,
    lambdacos=10,
    lambdaSWD=5,
    lambdalat=10,
    lambdarec=0.1,
    model_path=model_path,
    data_path=data_path,
    result_path=result_path,
    ot=False,
    device="cuda:5"
    )

In [13]:
K,cluster,emb_svg_B=model4.preprocess(svg_list,adata_sc, adata_ST,res=0.5)

Finding highly variable genes...
# overlap highly variable genes is: 1643
Normalizing and scaling...
AnnData object with n_obs × n_vars = 2695 × 32285
    obs: 'x', 'y', 'color', 'z', 'batch'
    var: 'gene_ids', 'feature_types', 'genome', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'hvg', 'log1p'
    obsm: 'spatial'
['Msra', 'Pvalb', 'Dlk2', 'Cdhr1', 'Kalrn', 'Tnnt1', 'Ptpro', 'Epop', 'Stard8', 'Elfn1', 'Vps16', 'Ivns1abp', 'Ermn', 'Elmod1', 'Lingo3', 'Stxbp6', 'Myl4', 'Nova1', 'AW551984', 'Gpr83', 'Garnl3', 'Rasgef1a', 'Dkkl1', 'Gria3', 'Nrn1', 'Lrrc10b', 'Gps2', 'Col6a1', 'Kctd3', 'Sv2c', 'Doc2g', 'Abl2', 'Folr1', 'Plekhn1', 'Limk2', 'Ndst4', 'Paxbp1', 'Sox1', 'Etl4', 'Ipcef1', 'Adamts19', 'Tshz1', 'Ptpru', 'Synpr', 'Rapgefl1', 'Egr3', 'Rae1', 'Plk2', 'AI593442', 'Nmb', 'Rarb', 'Arhgef9', 'Nxph3', 'Pde10a', 'Dock10', 'Neurod2', 'Unc13c', 'Grin3a', 'Chgb', 'Gpr101', 'Calb2', 'Dpysl5', 'Sox11', 'Cpne6', 'Kcnip2', 'Actn1', 'Tesc', 'Gng7', 

In [14]:
    #K,cluster=model4.preprocess(svg_list,adata_sc, adata_ST,res=0.5)
    training_idx_st=np.array(splits[0][0])
    model4.train(training_idx_rna,np.array([i for i in range((adata_ST.shape[0]))]))
    mu4,phi4,sigma4,z_A4, z_B4,m_A4,m_B4 = model4.eval2()

Begining time:  Sun Jul 12 19:39:28 2026
torch.Size([200, 23])
step 0, total_loss=1109.3447, loss_D=16.2933, loss_GAN=0.0000, loss_AE=56.3528, loss_cos=1.9547, loss_LA=45.2075, loss_SWD=14.0562
torch.Size([200, 23])
step 500, total_loss=22.0900, loss_D=0.0316, loss_GAN=0.0000, loss_AE=0.9269, loss_cos=0.4573, loss_LA=0.1913, loss_SWD=0.6237
torch.Size([200, 23])
step 1000, total_loss=14.9339, loss_D=0.0279, loss_GAN=0.0000, loss_AE=0.7033, loss_cos=0.2611, loss_LA=0.1109, loss_SWD=0.3480
step 1500, loss_lat=0.6366
step 2000, loss_lat=0.0243
step 2500, loss_lat=0.0307
Ending time:  Sun Jul 12 19:48:46 2026
Training takes 557.69 seconds
Localized scRNA-seq dataset has been saved!


In [15]:
x_input=np.concatenate((z_B4, emb_svg_B), axis=1)

In [ ]:
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
def compute_avg_neighbor_distance(loc, X, n_neighbors=10):
    """
    Compute for each spot the average distance (Euclidean) in cell-type proportion
    between the spot and its n_neighbors nearest neighbors in space.
    Parameters:
    loc : np.ndarray, shape (N, 2)
        Spatial coordinates of spots.
    X : np.ndarray, shape (N, D)
        Cell-type proportion matrix for each spot (D cell types).
    n_neighbors : int, default 10
        Number of nearest neighbors to consider.

    Returns:
    avg_distances : np.ndarray, shape (N,)
        Average distance for each spot.
    """
    N = loc.shape[0]
    # Find nearest neighbors (excluding self)
    nn = NearestNeighbors(n_neighbors=n_neighbors+1, metric='euclidean')
    nn.fit(loc)
    distances, indices = nn.kneighbors(loc)  # indices shape (N, n_neighbors+1), first is self

    # Exclude self (first column)
    neighbor_indices = indices[:, 1:]  # (N, n_neighbors)
    # neighbor_distances = distances[:, 1:]  # not needed

    avg_distances = np.zeros(N)
    for i in range(N):
        neighbors_X = X[neighbor_indices[i]]  # shape (n_neighbors, D)
        # Compute Euclidean distance between X[i] and each neighbor row
        sim = cosine_similarity([X[i]], neighbors_X)[0]  # shape (n_neighbors,)
        dists = 1 - sim   
        avg_distances[i] = np.mean(dists)
    return avg_distances

In [17]:
true_coord=adata_ST.obsm['spatial']
loc=true_coord
X=z_B4
avg_prop = compute_avg_neighbor_distance(loc, X, n_neighbors=15)

In [18]:
    y_true=np.zeros((true_coord.shape[0],1))
    y_true[selected_idx]=1

In [ ]:
scores =  avg_prop
labels = y_true  
adata_ST.obs['Abberant']=y_true
simulation_loc=pd.DataFrame(adata_ST.obsm['spatial'],columns=['x','y'],index=adata_ST.obs_names)
adata_ST.obs.to_csv('results/label_simulate.txt',sep='\t')
simulation_loc.to_csv('results/ST_location_simulate.txt',sep='\t')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
def weighted_mse_loss(pred, target, weights):
    """
    pred: (batch_size, 2)
    target: (batch_size, 2)
    weights: (batch_size,) 
    """
    sample_loss = torch.mean((pred - target) ** 2, dim=1)  # (batch_size,)
    weighted_loss = torch.mean(sample_loss * weights)
    return weighted_loss

class encoder_site(nn.Module):
    def __init__(self, n_input, n_latent):
        super(encoder_site, self).__init__()
        self.n_input = n_input
        self.n_latent = n_latent
        n_hidd_1 = 64
        n_hidd_2 = 32
        n_hidd_3 = 32
        n_hidd_4 = 8

        self.fc1 = nn.Linear(n_input, n_hidd_1)
        self.fc1_bn = nn.BatchNorm1d(n_hidd_1)
        self.fc2 = nn.Linear(n_hidd_1, n_hidd_2)
        self.fc2_bn = nn.BatchNorm1d(n_hidd_2)
        self.fc3 = nn.Linear(n_hidd_2, n_hidd_3)
        self.fc3_bn = nn.BatchNorm1d(n_hidd_3)
        self.fc4 = nn.Linear(n_hidd_3, n_hidd_4)
        self.fc4_bn = nn.BatchNorm1d(n_hidd_4)
        self.fc5 = nn.Linear(n_hidd_4, n_latent)

    def forward(self, input):
        h1 = F.relu(self.fc1_bn(self.fc1(input)))
        h2 = F.relu(self.fc2_bn(self.fc2(h1)))
        h3 = F.relu(self.fc3_bn(self.fc3(h2)))
        h4 = F.relu(self.fc4_bn(self.fc4(h3)))
        return self.fc5(h4)   


def train_model_with_weights(model, X, Y, train_idx, sample_weights=None, epochs=800, batch_size=64, lr=1e-3):
    """
    sample_weights: np.ndarray 
    """
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)


    X_train = X[train_idx]
    Y_train = Y[train_idx]


    if sample_weights is not None:
        if isinstance(sample_weights, np.ndarray):
            sample_weights = torch.tensor(sample_weights, dtype=torch.float32)
        W_train = sample_weights[train_idx]
    else:
        W_train = None

    dataset = TensorDataset(X_train, Y_train)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(epochs):
        total_loss = 0.0
        for batch_idx, (batch_x, batch_y) in enumerate(loader):
            if batch_x.shape[0]<=1:
                continue
            optimizer.zero_grad()
            pred = model(batch_x)

            if W_train is not None:
                # 
                start_idx = batch_idx * batch_size
                end_idx = min(start_idx + batch_size, len(train_idx))
                batch_weights = W_train[start_idx:end_idx]
                loss = weighted_mse_loss(pred, batch_y, batch_weights)
            else:
                #  MSE
                criterion = nn.MSELoss()
                loss = criterion(pred, batch_y)

            loss.backward()
            optimizer.step()
            total_loss += loss.item() * batch_x.size(0)

        if (epoch+1) % 100 == 0:
            avg_loss = total_loss / len(loader.dataset)
            print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.6f}")

    return model

sample_weights=avg_prop

aa=pd.DataFrame(y_true, columns=['label'])
aa['avg_prop']=avg_prop
label_simulate_sorted = aa.sort_values('avg_prop', ascending=False)

X_t = torch.tensor(x_input, dtype=torch.float32)
Y_t = torch.tensor(true_coord, dtype=torch.float32)
X_t=X_t[avg_prop<0.50,:]
Y_t=Y_t[avg_prop<0.50,:]

In [ ]:
model = encoder_site(n_input=X_t.shape[1], n_latent=Y_t.shape[1])
model = train_model_with_weights(
        model, X_t, Y_t, np.arange(X_t.shape[0]),
        sample_weights=None,  
        epochs=800,
        batch_size=64,
        lr=1e-2
    )

Epoch 100/800, Loss: 0.001118
Epoch 200/800, Loss: 0.002317
Epoch 300/800, Loss: 0.000805
Epoch 400/800, Loss: 0.000516
Epoch 500/800, Loss: 0.000622
Epoch 600/800, Loss: 0.000370
Epoch 700/800, Loss: 0.000673
Epoch 800/800, Loss: 0.000337


In [28]:
model.eval()
with torch.no_grad():
    final_pred = model(torch.tensor(x_input, dtype=torch.float32)).cpu().numpy()  # (len(val_idx), 2)

In [30]:
avg_prop0=avg_prop.copy()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
scores0 =  np.sum((final_pred-true_coord)**2,axis=1)**(1/2)
avg_prop_n=(avg_prop-np.min(avg_prop))/(np.max(avg_prop)-np.min(avg_prop))
scores0=(scores0-np.min(scores0))/(np.max(scores0)-np.min(scores0))
scores=scores0+0.5*avg_prop_n
labels = y_true    
fpr, tpr, thresholds_roc = roc_curve(labels, scores)
auroc = auc(fpr, tpr)

precision, recall, thresholds_pr = precision_recall_curve(labels, scores)
# average_precision_score 
aupr = average_precision_score(labels, scores)

print(f"AUROC (ROC ): {auroc:.4f}")
print(f"AUPR  (PR ): {aupr:.4f}")


plt.figure(figsize=(10, 4))

# ROC 
plt.subplot(1, 2, 1)
plt.plot(fpr, tpr, label=f'ROC (AUC = {auroc:.3f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('ROC Curve')
plt.legend()

# PR 
plt.subplot(1, 2, 2)
plt.plot(recall, precision, label=f'PR (AUPR = {aupr:.3f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
df_roc = pd.DataFrame({'fpr': fpr, 'tpr': tpr, 'threshold': thresholds_roc})
df_roc.to_csv('metrics/SCAD_ROC.txt', sep='\t', index=False)

precision_adj = precision[:-1]
recall_adj = recall[:-1]
df_pr = pd.DataFrame({'precision': precision_adj, 'recall': recall_adj, 'threshold': thresholds_pr})
df_pr.to_csv('metrics/SCAD_PR.txt', sep='\t', index=False)